### 02 — Build Silver Layer
**Project:** Adobe People Analytics — Semantic Layer Portfolio  
**Author:** Lani Mateo  
**Purpose:** Transform raw Bronze data into cleaned, typed dimension tables with SCD2 surrogate keys.  
In production, Data Engineering owns this layer. Built here to generate realistic data and demonstrate full-stack understanding.

**Key design decisions:**
- **Surrogate key:** `employee_sk` created via `ROW_NUMBER() OVER (ORDER BY emp_id, scd_start_date)`. Each SCD2 version gets a unique integer, converting many-to-many into clean one-to-many relationships in Power BI.
- **CAST vs TRY_CAST:** Uses CAST throughout because source data is machine-generated Python with deterministic types. Production pipelines from external sources should use TRY_CAST.
- **SCD2 integrity check:** Validates every emp_id has exactly one is_current=1 row. Pipeline stops if this check fails.

**Tables created:**
| Table | Rows | Description |
|-------|------|-------------|
| silver.dim_date | 4,748 | Day-grain date dimension 2018-2030 |
| silver.dim_department | 18 | Department reference with business unit and region |
| silver.dim_job | 23 | Job codes with job_level_sort for chart ordering |
| silver.dim_comp_band | 7 | Compensation bands with min/mid/max |
| silver.dim_employee | ~8,961 | SCD2 employee dimension with surrogate key |

**Run frequency:** After every Bronze refresh.  
**Validation:** SCD2 integrity check must show 0 duplicate current rows before proceeding to Gold.

In [0]:
-- =============================================================================
-- NOTEBOOK: 02_build_silver_layer
-- PURPOSE:  Build Silver dimension tables from Bronze raw data
--           Applies cleaning, typing, deduplication and SCD2 structure
-- CATALOG:  people_analytics
-- SCHEMA:   silver
-- AUTHOR:   Portfolio Project — Adobe People Analytics
-- DATE:     2026-04-09
-- =============================================================================

-- CELL 1 — Set catalog and schema
USE CATALOG people_analytics;
USE SCHEMA silver;

SELECT 'Silver layer build started' AS status, current_timestamp() AS run_ts;

In [0]:
-- =============================================================================
-- CELL 2 — dim_date
-- Source: bronze.dim_date_raw
-- Type:   Static dimension — no SCD needed
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.silver.dim_date
COMMENT 'Date spine 2018-2030. Static dimension, no SCD required.'
AS
SELECT
    CAST(date_sk        AS INT)     AS date_sk,
    CAST(date           AS DATE)    AS date,
    CAST(year           AS INT)     AS year,
    CAST(quarter        AS INT)     AS quarter,
    CAST(month          AS INT)     AS month,
    CAST(month_name     AS STRING)  AS month_name,
    CAST(month_short    AS STRING)  AS month_short,
    CAST(week           AS INT)     AS week,
    CAST(day            AS INT)     AS day,
    CAST(day_of_week    AS INT)     AS day_of_week,
    CAST(day_name       AS STRING)  AS day_name,
    CAST(is_weekday     AS INT)     AS is_weekday,
    CAST(is_month_end   AS INT)     AS is_month_end,
    CAST(is_quarter_end AS INT)     AS is_quarter_end,
    CAST(fiscal_year    AS INT)     AS fiscal_year,
    CAST(fiscal_quarter AS INT)     AS fiscal_quarter,
    CAST(fiscal_month   AS INT)     AS fiscal_month,
    CAST(yyyymm         AS INT)     AS yyyymm
FROM people_analytics.bronze.dim_date_raw
ORDER BY date_sk;

SELECT 'dim_date rows:' AS table_name, COUNT(*) AS row_count
FROM people_analytics.silver.dim_date;

In [0]:
-- =============================================================================
-- CELL 3 — dim_department
-- Source: bronze.employees_raw
-- Type:   SCD1 — latest version overwrites (dept attributes rarely change)
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.silver.dim_department
COMMENT 'Department dimension. SCD1 - latest values overwrite. One row per dept.'
AS
SELECT DISTINCT
    CAST(dept_id       AS STRING) AS dept_id,
    TRY_CAST(dept_name     AS STRING) AS dept_name,
    CAST(business_unit AS STRING) AS business_unit,
    CAST(cost_centre   AS STRING) AS cost_centre,
    CAST(location      AS STRING) AS location,
    CAST(region        AS STRING) AS region,
    current_timestamp()           AS load_ts
FROM people_analytics.bronze.employees_raw
WHERE dept_id IS NOT NULL
  AND dept_name IS NOT NULL
ORDER BY dept_id;

SELECT 'dim_department rows:' AS table_name, COUNT(*) AS row_count
FROM people_analytics.silver.dim_department;

In [0]:
-- =============================================================================
-- CELL 4 — dim_job
-- Source: bronze.employees_raw
-- Type:   SCD1 — latest values overwrite
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.silver.dim_job
COMMENT 'Job dimension. SCD1 - one row per job code.'
AS
SELECT DISTINCT
    CAST(job_code         AS STRING)  AS job_code,
    CAST(job_title        AS STRING)  AS job_title,
    CAST(job_family       AS STRING)  AS job_family,
    CAST(job_level        AS STRING)  AS job_level,
    CAST(is_people_manager AS BOOLEAN) AS is_people_manager,
    CASE job_level
        WHEN 'IC1' THEN 1 WHEN 'IC2' THEN 2 WHEN 'IC3' THEN 3
        WHEN 'IC4' THEN 4 WHEN 'IC5' THEN 5 WHEN 'M1'  THEN 6
        WHEN 'M2'  THEN 7 WHEN 'M3'  THEN 8 ELSE 0
    END                               AS job_level_sort,
    CASE
        WHEN job_level IN ('IC1','IC2','IC3','IC4','IC5') THEN 'Individual Contributor'
        WHEN job_level IN ('M1','M2','M3')                THEN 'People Manager'
        ELSE 'Unknown'
    END                               AS job_level_category,
    current_timestamp()               AS load_ts
FROM people_analytics.bronze.employees_raw
WHERE job_code IS NOT NULL
ORDER BY job_family, job_level_sort;

SELECT 'dim_job rows:' AS table_name, COUNT(*) AS row_count
FROM people_analytics.silver.dim_job;

In [0]:
-- =============================================================================
-- CELL 5 — dim_comp_band
-- Source: Hardcoded reference data (stable, defined by Total Rewards team)
-- Type:   SCD1
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.silver.dim_comp_band
COMMENT 'Compensation band reference. Defined by Total Rewards. SCD1.'
AS
SELECT
    band_name,
    band_min,
    band_mid,
    band_max,
    ROUND((band_max - band_min) / band_min * 100, 1) AS band_range_pct,
    current_timestamp() AS load_ts
FROM (
    VALUES
        ('Band 1 (IC1)',  65000,   90000,  115000),
        ('Band 2 (IC2)',  85000,  115000,  150000),
        ('Band 3 (IC3)', 110000,  150000,  190000),
        ('Band 4 (IC4)', 150000,  195000,  240000),
        ('Band 5 (IC5)', 200000,  255000,  310000),
        ('Band 6 (M1)',  170000,  220000,  270000),
        ('Band 7 (M2+)', 220000,  285000,  350000)
) AS t(band_name, band_min, band_mid, band_max);

SELECT 'dim_comp_band rows:' AS table_name, COUNT(*) AS row_count
FROM people_analytics.silver.dim_comp_band;

In [0]:
-- =============================================================================
-- CELL 6 — dim_employee (SCD2)
-- Source: bronze.employees_raw
-- Type:   SCD2 — full history preserved with surrogate key
--
-- KEY DESIGN DECISIONS:
--   employee_sk  = surrogate key, unique per SCD2 version
--   emp_id       = natural key, same across all versions
--   scd_start_date / scd_end_date = version validity window
--   is_current   = 1 for the active version, 0 for historical
--
-- Power BI usage:
--   Current headcount  → filter is_current = 1 AND status = 'Active'
--   Historical reports → join fact tables on employee_sk (captured at time)
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.silver.dim_employee
COMMENT 'Employee dimension with full SCD2 history. One row per employee version.
Use is_current=1 for current headcount. Join fact tables on employee_sk for history.'
AS
SELECT
    -- Surrogate key: unique per SCD2 version
    ROW_NUMBER() OVER (
        ORDER BY emp_id, TRY_CAST(scd_start_date AS TIMESTAMP)
    )                                               AS employee_sk,

    -- Natural key: stable across all versions
    CAST(emp_id             AS STRING)              AS emp_id,
    CAST(version            AS INT)                 AS version,

    -- Personal attributes (never change across versions)
    TRIM(CAST(full_name     AS STRING))             AS full_name,
    CAST(gender             AS STRING)              AS gender,
    CAST(ethnicity          AS STRING)              AS ethnicity,
    CAST(birth_date         AS DATE)                AS birth_date,
    CAST(emp_type           AS STRING)              AS employment_type,
    CAST(hire_date          AS DATE)                AS hire_date,

    -- Org attributes (change triggers new SCD2 version)
    CAST(dept_id            AS STRING)              AS dept_id,
    TRIM(CAST(dept_name     AS STRING))             AS dept_name,
    CAST(business_unit      AS STRING)              AS business_unit,
    CAST(location           AS STRING)              AS location,
    CAST(region             AS STRING)              AS region,

    -- Job attributes (change triggers new SCD2 version)
    CAST(job_code           AS STRING)              AS job_code,
    CAST(job_title          AS STRING)              AS job_title,
    CAST(job_family         AS STRING)              AS job_family,
    CAST(job_level          AS STRING)              AS job_level,
    CAST(is_people_manager  AS BOOLEAN)             AS is_people_manager,

    -- Compensation (change triggers new SCD2 version)
    CAST(salary             AS DECIMAL(18,2))       AS salary,
    CAST(comp_band          AS STRING)              AS comp_band,

    -- Derived compensation metrics
    CASE comp_band
        WHEN 'Band 1 (IC1)' THEN ROUND(CAST(salary AS DECIMAL) / 90000,  3)
        WHEN 'Band 2 (IC2)' THEN ROUND(CAST(salary AS DECIMAL) / 115000, 3)
        WHEN 'Band 3 (IC3)' THEN ROUND(CAST(salary AS DECIMAL) / 150000, 3)
        WHEN 'Band 4 (IC4)' THEN ROUND(CAST(salary AS DECIMAL) / 195000, 3)
        WHEN 'Band 5 (IC5)' THEN ROUND(CAST(salary AS DECIMAL) / 255000, 3)
        WHEN 'Band 6 (M1)'  THEN ROUND(CAST(salary AS DECIMAL) / 220000, 3)
        WHEN 'Band 7 (M2+)' THEN ROUND(CAST(salary AS DECIMAL) / 285000, 3)
        ELSE NULL
    END                                             AS compa_ratio,

    -- Manager (change triggers new SCD2 version)
    CAST(manager_id         AS STRING)              AS manager_id,

    -- Employment status
    CAST(status             AS STRING)              AS status,
    CAST(termination_date   AS DATE)                AS termination_date,
    CAST(termination_reason AS STRING)              AS termination_reason,

    -- Derived termination type
    CASE
        WHEN termination_reason LIKE 'Involuntary%' THEN 'Involuntary'
        WHEN termination_reason = 'Retirement'      THEN 'Retirement'
        WHEN termination_reason IS NOT NULL         THEN 'Voluntary'
        ELSE NULL
    END                                             AS termination_type,

    -- SCD2 validity columns
    CAST(scd_start_date     AS DATE)                AS scd_start_date,
    CAST(scd_end_date       AS DATE)                AS scd_end_date,
    CAST(is_current         AS INT)                 AS is_current,

    -- Audit
    current_timestamp()                             AS load_ts

FROM people_analytics.bronze.employees_raw
WHERE emp_id IS NOT NULL;

SELECT 'dim_employee rows:' AS table_name, COUNT(*) AS row_count
FROM people_analytics.silver.dim_employee;

In [0]:
-- =============================================================================
-- CELL 7 — Verify Silver layer
-- =============================================================================

-- Row count summary
SELECT 'dim_date'       AS silver_table, COUNT(*) AS rows FROM people_analytics.silver.dim_date
UNION ALL
SELECT 'dim_department' AS silver_table, COUNT(*) AS rows FROM people_analytics.silver.dim_department
UNION ALL
SELECT 'dim_job'        AS silver_table, COUNT(*) AS rows FROM people_analytics.silver.dim_job
UNION ALL
SELECT 'dim_comp_band'  AS silver_table, COUNT(*) AS rows FROM people_analytics.silver.dim_comp_band
UNION ALL
SELECT 'dim_employee'   AS silver_table, COUNT(*) AS rows FROM people_analytics.silver.dim_employee
ORDER BY silver_table;

In [0]:
-- =============================================================================
-- CELL 8 — SCD2 integrity check
-- Every emp_id must have exactly one is_current = 1 row
-- =============================================================================

SELECT
    'SCD2 integrity check' AS check_name,
    COUNT(*) AS employees_with_duplicate_current,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
    SELECT emp_id
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1
    GROUP BY emp_id
    HAVING COUNT(*) > 1
) violations;

In [0]:
-- =============================================================================
-- CELL 9 — SCD2 version summary
-- =============================================================================

SELECT
    version,
    COUNT(*)                AS employee_count,
    ROUND(COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (), 1) AS pct_of_total
FROM people_analytics.silver.dim_employee
WHERE is_current = 1
GROUP BY version
ORDER BY version;

In [0]:
-- =============================================================================
-- CELL 10 — Sample employee history (SCD2 demo)
-- Shows a single employee across all their versions
-- =============================================================================

SELECT
    emp_id,
    full_name,
    version,
    dept_name,
    job_title,
    job_level,
    salary,
    compa_ratio,
    status,
    scd_start_date,
    scd_end_date,
    is_current
FROM people_analytics.silver.dim_employee
WHERE emp_id IN (
    -- Find employees with 3+ versions to show rich SCD2 history
    SELECT emp_id
    FROM people_analytics.silver.dim_employee
    GROUP BY emp_id
    HAVING COUNT(*) >= 3
    LIMIT 1
)
ORDER BY emp_id, version;

In [0]:
-- =============================================================================
-- CELL 11 — Active headcount by business unit
-- Smoke test: confirms Silver data can answer business questions
-- =============================================================================

SELECT
    business_unit,
    COUNT(*)                          AS active_headcount,
    ROUND(AVG(salary), 0)             AS avg_salary,
    ROUND(AVG(compa_ratio), 3)        AS avg_compa_ratio,
    SUM(CASE WHEN is_people_manager
             THEN 1 ELSE 0 END)       AS people_managers
FROM people_analytics.silver.dim_employee
WHERE is_current = 1
  AND status     = 'Active'
GROUP BY business_unit
ORDER BY active_headcount DESC;

SELECT 'Silver layer build complete. Ready for Gold layer.' AS status;